## Imports

In [1]:

import tensorflow as tf
import os
import sys


from preprocess import create_data_generators, get_class_map

# import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau,ModelCheckpoint

from pathlib import Path

## Constants


In [2]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20

## Paths

In [3]:
from pathlib import Path
import os

PROJECT_BASE_DIR = Path(os.environ["NEURODETECT_BASE_DIR"])

TRAIN_DIR = str(PROJECT_BASE_DIR / "train")
TEST_DIR = str(PROJECT_BASE_DIR / "test")
MODELS_DIR = PROJECT_BASE_DIR / "models"

print("Project base:", PROJECT_BASE_DIR)
print("Train path:", TRAIN_DIR)
print("Test path:", TEST_DIR)
print("Models path:", MODELS_DIR)

print("Train exists:", os.path.exists(TRAIN_DIR))
print("Test exists:", os.path.exists(TEST_DIR))
print("Models exists:", os.path.exists(MODELS_DIR))

Project base: G:\My Drive\SeniorProject\datasets\dataset
Train path: G:\My Drive\SeniorProject\datasets\dataset\train
Test path: G:\My Drive\SeniorProject\datasets\dataset\test
Models path: G:\My Drive\SeniorProject\datasets\dataset\models
Train exists: True
Test exists: True
Models exists: True


In [4]:
import shutil
from pathlib import Path

SECOND_DATASET_TRAIN = Path(r"G:\My Drive\SeniorProject\datasets\dataset2\train")
SECOND_DATASET_TEST  = Path(r"G:\My Drive\SeniorProject\datasets\dataset2\test")

def merge_into(source_dir, dest_dir):
    for class_folder in source_dir.iterdir():
        if class_folder.is_dir():
            dest = Path(dest_dir) / class_folder.name
            dest.mkdir(exist_ok=True)
            copied = 0
            skipped = 0
            for img in class_folder.glob("*.*"):
                try:
                    shutil.copy(img, dest / img.name)
                    copied += 1
                except OSError as e:
                    print(f"    Skipped {img.name}: {e}")
                    skipped+=1
            print(f"  {class_folder.name}: copied {copied} images")

print("Merging train...")
merge_into(SECOND_DATASET_TRAIN, TRAIN_DIR)

print("\nMerging test...")
merge_into(SECOND_DATASET_TEST, TEST_DIR)

print("\nFinal counts:")
for folder in Path(TRAIN_DIR).iterdir():
    if folder.is_dir():
        count = len(list(folder.glob("*.*")))
        print(f"  {folder.name}: {count} images")

Merging train...
  glioma_tumor: copied 1401 images
  meningioma_tumor: copied 1401 images
  no_tumor: copied 1401 images
  pituitary_tumor: copied 1401 images

Merging test...
  glioma_tumor: copied 401 images
  meningioma_tumor: copied 401 images
  no_tumor: copied 401 images
  pituitary_tumor: copied 401 images

Final counts:
  pituitary_tumor: 2228 images
  no_tumor: 1796 images
  glioma_tumor: 2227 images
  meningioma_tumor: 2223 images


## Train Gen

In [5]:
train_generator, test_generator = create_data_generators(
    TRAIN_DIR,
    TEST_DIR,
    IMG_SIZE,
    BATCH_SIZE
)

labels = get_class_map(train_generator)

Found 8470 images belonging to 4 classes.
Found 1994 images belonging to 4 classes.


## Batch

In [6]:
X_batch, y_batch = next(train_generator) 
print("Batch image shape:", X_batch.shape)  

print("\nLoading Test Data:")

X_batch, y_batch = next(test_generator)  
print("Batch image shape:", X_batch.shape) 

Batch image shape: (32, 224, 224, 3)

Loading Test Data:
Batch image shape: (32, 224, 224, 3)


#### commented because not needed ↓

## Baseline Model

In [7]:
# num_classes = len(labels)

# model_baseline = tf.keras.Sequential([
#     tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
#     tf.keras.layers.MaxPooling2D(2,2),

#     tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
#     tf.keras.layers.MaxPooling2D(2,2),

#     tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
#     tf.keras.layers.MaxPooling2D(2,2),

#     tf.keras.layers.Flatten(),

#     tf.keras.layers.Dense(128, activation='relu'),
#     tf.keras.layers.Dropout(0.5),

#     tf.keras.layers.Dense(num_classes, activation='softmax')
# ])

# model_baseline.summary()

## Compile

In [8]:
# model_baseline.compile(
#     optimizer='adam',
#     loss='categorical_crossentropy',
#     metrics=['accuracy']
# )

## Run Epochs

In [9]:
# import time
# import math

# steps_per_epoch = math.ceil(train_generator.samples / BATCH_SIZE)
# validation_steps = math.ceil(test_generator.samples / BATCH_SIZE)

# start = time.time()
# history_baseline = model_baseline.fit(
#     train_generator,
#     steps_per_epoch=steps_per_epoch,
#     validation_data=test_generator,
#     validation_steps=validation_steps,
#     epochs=EPOCHS
# )
# train_time_sec = time.time() - start

# print("Training time (sec):", round(train_time_sec, 2))

## Results

In [10]:
# baseline_loss, baseline_acc = model_baseline.evaluate(
#     test_generator,
#     steps=validation_steps
# )
# print("Baseline Loss:", baseline_loss)
# print("Baseline Accuracy:", baseline_acc)

## Analysis

In [11]:
# import numpy as np
# from sklearn.metrics import confusion_matrix, classification_report

# # reset generator to start
# test_generator.reset()

# y_prob = model_baseline.predict(test_generator, steps=validation_steps)
# y_pred = np.argmax(y_prob, axis=1)

# y_true = test_generator.classes  # integer labels in directory order
# class_names = list(test_generator.class_indices.keys())

# cm = confusion_matrix(y_true, y_pred)
# print("Confusion Matrix:\n", cm)

# print("\nClassification Report:\n")
# print(classification_report(y_true, y_pred, target_names=class_names))

## Save Model

In [12]:
# os.makedirs(MODELS_DIR, exist_ok=True)

# baseline_model_path = MODELS_DIR / "baseline_cnn.keras"
# model_baseline.save(str(baseline_model_path))
# print(f"Saved to {baseline_model_path}")

# Transfer Model

#### Phase 1

In [13]:

transfer_train_datagen= ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.85,1.15],
    fill_mode='nearest'
)
transfer_test_datagen=ImageDataGenerator(
    preprocessing_function=preprocess_input
)
train_gen_tf=transfer_train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)
test_gen_tf=transfer_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 8470 images belonging to 4 classes.
Found 1994 images belonging to 4 classes.


## Model

In [14]:
base_model=MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

X= base_model.output
X=GlobalAveragePooling2D()(X)
X=Dense(256, activation='relu')(X)
X=Dropout(0.4)(X)
X=Dense(128,activation='relu')(X)
X=Dropout(0.3)(X)
outputs= Dense(train_gen_tf.num_classes, activation='softmax')(X)
model_transfer=Model(inputs=base_model.input, outputs=outputs)

## Compile

In [15]:
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase1=[
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_1r=1e-6),
    ModelCheckpoint(
        str(MODELS_DIR / 'hopefully_best_transfer_phase1.keras'),
        monitor='val_accuracy', save_best_only=True
    )
]

## Epochs

In [16]:
EPOCHS_TL=20
history_transfer= model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=EPOCHS_TL,
    callbacks=callbacks_phase1
)

Epoch 1/20
265/265 ━━━━━━━━━━━━━━━━━━━━ 2143s 8s/step - accuracy: 0.7677 - loss: 0.6050 - val_accuracy: 0.7523 - val_loss: 0.7644 - learning_rate: 0.0010
Epoch 2/20
265/265 ━━━━━━━━━━━━━━━━━━━━ 294s 1s/step - accuracy: 0.8473 - loss: 0.4023 - val_accuracy: 0.7844 - val_loss: 0.5873 - learning_rate: 0.0010
Epoch 3/20
265/265 ━━━━━━━━━━━━━━━━━━━━ 409s 2s/step - accuracy: 0.8712 - loss: 0.3473 - val_accuracy: 0.7959 - val_loss: 0.5920 - learning_rate: 0.0010
Epoch 4/20
265/265 ━━━━━━━━━━━━━━━━━━━━ 276s 1s/step - accuracy: 0.8753 - loss: 0.3322 - val_accuracy: 0.7999 - val_loss: 0.6466 - learning_rate: 0.0010
Epoch 5/20
265/265 ━━━━━━━━━━━━━━━━━━━━ 274s 1s/step - accuracy: 0.8854 - loss: 0.2979 - val_accuracy: 0.8039 - val_loss: 0.5960 - learning_rate: 0.0010
Epoch 6/20
265/265 ━━━━━━━━━━━━━━━━━━━━ 279s 1s/step - accuracy: 0.9024 - loss: 0.2521 - val_accuracy: 0.8220 - val_loss: 0.6467 - learning_rate: 5.0000e-04
Epoch 7/20
265/265 ━━━━━━━━━━━━━━━━━━━━ 294s 1s/step - accuracy: 0.9139 - los

## Results

In [17]:
transfer_loss, transfer_acc = model_transfer.evaluate(test_gen_tf)
print("Transfer Learning Loss:", transfer_loss)

63/63 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.8716 - loss: 0.5816
Transfer Learning Loss: 0.581561267375946


## Comparision

In [18]:
# print('Baseline Accuracy:', baseline_acc)
print('Transfer Learning Accuracy:', transfer_acc)

Transfer Learning Accuracy: 0.8716148734092712


In [19]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
historical_finetune= model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=10
)

Epoch 1/10
265/265 ━━━━━━━━━━━━━━━━━━━━ 1229s 5s/step - accuracy: 0.8112 - loss: 0.6728 - val_accuracy: 0.8350 - val_loss: 0.9196
Epoch 2/10
265/265 ━━━━━━━━━━━━━━━━━━━━ 613s 2s/step - accuracy: 0.8850 - loss: 0.3212 - val_accuracy: 0.8405 - val_loss: 0.9139
Epoch 3/10
265/265 ━━━━━━━━━━━━━━━━━━━━ 591s 2s/step - accuracy: 0.8919 - loss: 0.2867 - val_accuracy: 0.8501 - val_loss: 0.8317
Epoch 4/10
265/265 ━━━━━━━━━━━━━━━━━━━━ 556s 2s/step - accuracy: 0.9182 - loss: 0.2239 - val_accuracy: 0.8571 - val_loss: 0.7434
Epoch 5/10
265/265 ━━━━━━━━━━━━━━━━━━━━ 436s 2s/step - accuracy: 0.9218 - loss: 0.2006 - val_accuracy: 0.8591 - val_loss: 0.7611
Epoch 6/10
265/265 ━━━━━━━━━━━━━━━━━━━━ 803s 3s/step - accuracy: 0.9327 - loss: 0.1855 - val_accuracy: 0.8601 - val_loss: 0.7095
Epoch 7/10
265/265 ━━━━━━━━━━━━━━━━━━━━ 748s 3s/step - accuracy: 0.9368 - loss: 0.1669 - val_accuracy: 0.8741 - val_loss: 0.6674
Epoch 8/10
265/265 ━━━━━━━━━━━━━━━━━━━━ 912s 3s/step - accuracy: 0.9412 - loss: 0.1576 - val_acc

## Save Model

In [23]:
os.makedirs(MODELS_DIR, exist_ok=True)

transfer_model_path = MODELS_DIR / "mobilenetv2_transfer.keras"
model_transfer.save(str(transfer_model_path))
print(f"Saved to {transfer_model_path}")

Saved to G:\My Drive\SeniorProject\datasets\dataset\models\mobilenetv2_transfer.keras


#### Phase 2

### Callbacks

In [24]:
callbacks_phase2 = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
    ModelCheckpoint(
        str(MODELS_DIR / 'best_transfer_phase2.keras'),
        monitor='val_accuracy', save_best_only=True
    )
]

### Unfreeze + Recompile

In [25]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

### Run

In [26]:
history_finetune = model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=20,
    callbacks=callbacks_phase2
)

Epoch 1/20
265/265 ━━━━━━━━━━━━━━━━━━━━ 379s 1s/step - accuracy: 0.9515 - loss: 0.1280 - val_accuracy: 0.8887 - val_loss: 0.6313 - learning_rate: 1.0000e-05
Epoch 2/20
265/265 ━━━━━━━━━━━━━━━━━━━━ 335s 1s/step - accuracy: 0.9579 - loss: 0.1101 - val_accuracy: 0.8932 - val_loss: 0.6181 - learning_rate: 1.0000e-05
Epoch 3/20
265/265 ━━━━━━━━━━━━━━━━━━━━ 301s 1s/step - accuracy: 0.9623 - loss: 0.1056 - val_accuracy: 0.8987 - val_loss: 0.5944 - learning_rate: 1.0000e-05
Epoch 4/20
265/265 ━━━━━━━━━━━━━━━━━━━━ 298s 1s/step - accuracy: 0.9651 - loss: 0.1038 - val_accuracy: 0.8997 - val_loss: 0.6113 - learning_rate: 1.0000e-05
Epoch 5/20
265/265 ━━━━━━━━━━━━━━━━━━━━ 293s 1s/step - accuracy: 0.9661 - loss: 0.0934 - val_accuracy: 0.8982 - val_loss: 0.6541 - learning_rate: 1.0000e-05
Epoch 6/20
265/265 ━━━━━━━━━━━━━━━━━━━━ 1849s 7s/step - accuracy: 0.9671 - loss: 0.0898 - val_accuracy: 0.9062 - val_loss: 0.5831 - learning_rate: 1.0000e-05
Epoch 7/20
265/265 ━━━━━━━━━━━━━━━━━━━━ 363s 1s/step - ac

### Compare

In [27]:
test_gen_tf.reset()
finetune_loss, finetune_acc = model_transfer.evaluate(test_gen_tf)
print(f"Phase 1 Accuracy:  {transfer_acc:.4f}")
print(f"Phase 2 Accuracy:  {finetune_acc:.4f}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 37s 588ms/step - accuracy: 0.9188 - loss: 0.5774
Phase 1 Accuracy:  0.8716
Phase 2 Accuracy:  0.9188


### Save

In [28]:
os.makedirs(MODELS_DIR, exist_ok=True)
transfer_model_path = MODELS_DIR / "mobilenetv2_finetuned.keras"
model_transfer.save(str(transfer_model_path))
print(f"Saved to {transfer_model_path}")

Saved to G:\My Drive\SeniorProject\datasets\dataset\models\mobilenetv2_finetuned.keras


# Phase 3

## Callbacks

In [ ]:
callbacks_phase3 = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-8),
    ModelCheckpoint(
        str(MODELS_DIR / 'best_transfer_phase3.keras'),
        monitor='val_accuracy', save_best_only=True
    )
]

## Unfreeze more Layers 

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-60]: 
    layer.trainable = False

model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6), 
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Run Epochs

In [ ]:
history_phase3 = model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=30,  
    callbacks=callbacks_phase3
)

## Evaluate Improvements

In [ ]:
test_gen_tf.reset()
phase3_loss, phase3_acc = model_transfer.evaluate(test_gen_tf)
print(f"Phase 1 Accuracy: {transfer_acc:.4f}")
print(f"Phase 2 Accuracy: {finetune_acc:.4f}")
print(f"Phase 3 Accuracy: {phase3_acc:.4f}")